# 第八章：大语言模型 (LLM) 与对齐技术

## 从 Transformer 到 RLHF——理解 ChatGPT 背后的技术栈

2022 年底 ChatGPT 横空出世，改变了 AI 发展轨迹。本章从 Transformer 架构出发，
介绍预训练、监督微调 (SFT)、人类反馈强化学习 (RLHF) 等核心技术，
让你理解现代 LLM 的工作原理和训练范式。


## 8.1 Transformer 架构

2017 年 Google 在论文 *"Attention Is All You Need"* 中提出 Transformer，
它彻底抛弃了 RNN/LSTM 的序列递归结构，完全基于**自注意力机制 (Self-Attention)**。

### 整体架构

```
输入序列 → [Token Embedding + Positional Encoding]
                ↓
    ┌───────────────────────────┐
    │  Multi-Head Self-Attention │ (× N 层)
    │  Add & Norm                │
    │  Feed-Forward Network      │
    │  Add & Norm                │
    └───────────────────────────┘
                ↓
           输出 (Decoder 为自回归生成)
```

### 自注意力：核心公式

对于输入序列 $\mathbf{X} \in \mathbb{R}^{n \times d}$（$n$ 个 token，$d$ 维），通过三个可学习矩阵投影：

$$
\mathbf{Q} = \mathbf{X}\mathbf{W}^Q, \quad \mathbf{K} = \mathbf{X}\mathbf{W}^K, \quad \mathbf{V} = \mathbf{X}\mathbf{W}^V
$$

**缩放点积注意力 (Scaled Dot-Product Attention)：**

$$
\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^T}{\sqrt{d_k}}\right) \mathbf{V}
$$

- $\mathbf{Q}\mathbf{K}^T$：计算 query 和 key 的**相似度矩阵**（注意力权重）
- $\sqrt{d_k}$：缩放因子，防止点积过大导致 softmax 梯度消失
- $\times \mathbf{V}$：加权聚合 value 向量

### 多头注意力 (Multi-Head Attention)

单头注意力只能关注一种关系模式。多头注意力并行运行多个独立的注意力"头"，
每个头学习不同的关系模式：

$$
\text{MultiHead}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) \mathbf{W}^O
$$
$$
\text{head}_i = \text{Attention}(\mathbf{Q}\mathbf{W}^Q_i, \mathbf{K}\mathbf{W}^K_i, \mathbf{V}\mathbf{W}^V_i)
$$


### 8.1.1 为什么 Attention 有效：信息检索的视角

Attention 机制可以理解为一个**可微分的键值检索系统**。

**类比：图书馆查资料**

你去图书馆找关于"梯度下降"的书——
- **Query (Q)**：你的查询词"梯度下降"
- **Key (K)**：每本书的索引标签
- **Value (V)**：每本书的实际内容

系统计算 Query 与每个 Key 的**相似度**（注意力权重），然后用这些权重对 Value 做**加权求和**——最相关的书贡献最多内容。

在 Transformer 中，Q、K、V 都来自同一条输入序列（Self-Attention），所以每个 token 都在"查询"序列中所有其他 token：
$$
\text{Output}_i = \sum_{j=1}^{n} \underbrace{\text{softmax}\left(\frac{q_i^T k_j}{\sqrt{d_k}}\right)}_{\text{token i 对 token j 的关注度}} \cdot v_j
$$

#### 为什么除以 $\sqrt{d_k}$？

当 $d_k$ 很大时（如 64 或 128），点积 $q_i^T k_j$ 的值会很大——因为它是 $d_k$ 个随机变量的和，方差约为 $d_k$。

大点积值导致 softmax 输出趋向 one-hot（只有最大的那个接近 1，其余接近 0）→ 梯度趋近于 0 → 训练不动。

除以 $\sqrt{d_k}$ 将方差缩放回 1，保持 softmax 输出的平滑性，让梯度能正常流动。

#### Self-Attention vs Cross-Attention

| 类型 | Q 来源 | K,V 来源 | 用途 |
|------|--------|---------|------|
| **Self-Attention** | 当前序列 | 当前序列 | 捕捉序列内部关系（GPT 的基础） |
| **Cross-Attention** | 当前序列 | 另一序列 | 翻译（源语言 → 目标语言）、多模态 |

GPT 只使用因果 Self-Attention，而原始 Transformer 的 Decoder 同时使用两种。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np

# === 从零实现缩放点积注意力 ===
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q, K, V: (batch, n_heads, seq_len, d_k)
    """
    d_k = Q.size(-1)
    
    # scores: (batch, n_heads, seq_len, seq_len)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    
    # 可选的因果掩码（decoder 防止看到未来 token）
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    attn_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, V)
    return output, attn_weights

# 演示：一个小序列的自注意力
batch, n_heads, seq_len, d_k = 1, 2, 5, 4
Q = torch.randn(batch, n_heads, seq_len, d_k)
K = torch.randn(batch, n_heads, seq_len, d_k)
V = torch.randn(batch, n_heads, seq_len, d_k)

output, attn = scaled_dot_product_attention(Q, K, V)
print(f"输出 shape: {output.shape}")   # (1, 2, 5, 4)
print(f"注意力矩阵 shape: {attn.shape}")  # (1, 2, 5, 5)
print(f"注意力权重 (head 0):\n{attn[0, 0].detach().numpy().round(3)}")
print(f"每行和: {attn[0, 0].sum(-1).detach().numpy()}")  # 应该都是 1.0


### 位置编码 (Positional Encoding)

Transformer 没有循环结构，无法感知 token 的**顺序**。位置编码为每个位置注入唯一的位置信息。

原始论文使用三角函数位置编码：

$$
PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right)
$$
$$
PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)
$$

现代 LLM 多使用**可学习位置编码**（如 GPT）或 **RoPE (Rotary Position Embedding)**。


In [ ]:
# === Sinusoidal Positional Encoding ===
def sinusoidal_positional_encoding(seq_len, d_model):
    pe = torch.zeros(seq_len, d_model)
    position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                         (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

pe = sinusoidal_positional_encoding(50, 64)

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))
plt.imshow(pe.numpy(), aspect='auto', cmap='RdBu')
plt.colorbar(label='Encoding Value')
plt.xlabel('Embedding Dimension'); plt.ylabel('Position')
plt.title('Sinusoidal Positional Encoding (50 positions, 64 dims)')
plt.savefig('../fig/positional_encoding.png', dpi=100, bbox_inches='tight')
plt.show()


## 8.2 GPT 系列：从 GPT-1 到 GPT-4

| 模型 | 年份 | 参数量 | 核心创新 |
|------|------|--------|---------|
| **GPT-1** | 2018 | 117M | 无监督预训练 + 有监督微调 |
| **GPT-2** | 2019 | 1.5B | Zero-shot 能力，认为足够大的语言模型可直接完成任务 |
| **GPT-3** | 2020 | 175B | In-context learning, few-shot prompting |
| **GPT-4** | 2023 | ~1.8T (估计) | 多模态，更强的推理能力 |
| **ChatGPT** | 2022 | GPT-3.5 + RLHF | 指令遵循 + 对话能力 |

### 预训练：Next Token Prediction

GPT 的训练目标极其简单——**预测下一个 token**：

$$
\mathcal{L}_{\text{pretrain}} = -\sum_{t=1}^{T} \log P(x_t | x_{<t})
$$

给定前文 $x_{<t}$，模型需要预测 $x_t$。这个看似简单的任务，
在足够大的模型和海量数据上，迫使模型学会语法、事实知识、推理能力。

### GPT 的核心：因果自注意力 (Causal Self-Attention)

与原始 Transformer 不同，GPT 使用**因果掩码 (Causal Mask)**，确保每个 token 只能看到**它之前**的 token（不能偷看未来）。


In [ ]:
# === 因果掩码可视化 ===
seq_len = 8
mask = torch.tril(torch.ones(seq_len, seq_len))  # 下三角矩阵

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(mask, cmap='Blues')
for i in range(seq_len):
    for j in range(seq_len):
        ax.text(j, i, '✓' if mask[i, j] else '✗', ha='center', va='center',
                color='green' if mask[i, j] else 'red', fontsize=10)
ax.set_xlabel('Key Position (attended from)')
ax.set_ylabel('Query Position (can attend to)')
ax.set_title('Causal Attention Mask\nToken t can only attend to tokens ≤ t')
plt.tight_layout()
plt.savefig('../fig/causal_mask.png', dpi=100, bbox_inches='tight')
plt.show()


## 8.3 监督微调 (Supervised Fine-Tuning, SFT)

预训练的 LLM 是一个"文字接龙"机器——它擅长续写，但**不擅长遵循指令**。
SFT 通过在（指令，回答）数据集上继续训练，让模型学会"听懂人话"。

### SFT 流程

1. 收集高质量（指令，回答）对（人工标注或蒸馏）
2. 将指令格式化为固定模板（如 ChatML）
3. 在预训练模型上继续训练（通常 2-5 epochs）
4. **仅计算答案部分的损失**，指令部分不参与 loss 计算

### 损失掩码 (Loss Masking)

```
输入:  <|user|>解释一下什么是反向传播<|assistant|>反向传播是一种...
损失:  0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1
                ↑ 仅在这些位置计算损失
```

### 核心思想

SFT 的本质是**行为克隆 (Behavioral Cloning)**——让模型模仿人类给出的高质量回答。
简单但有效：一个经过良好 SFT 的 7B 模型可以媲美未对齐的 70B 模型。


In [ ]:
# === SFT 训练数据格式示例 ===
# 使用 HuggingFace datasets 加载真实 SFT 数据集
# 注意：需要安装: pip install datasets

# 模拟 SFT 数据格式
sft_example = {
    "messages": [
        {"role": "user", "content": "用 Python 实现快速排序"},
        {"role": "assistant", "content": "def quicksort(arr):\n    if len(arr) <= 1:\n        return arr\n    pivot = arr[len(arr)//2]\n    left = [x for x in arr if x < pivot]\n    middle = [x for x in arr if x == pivot]\n    right = [x for x in arr if x > pivot]\n    return quicksort(left) + middle + quicksort(right)"}
    ]
}

# 格式化函数
def format_chatml(messages):
    """ChatML 格式"""
    formatted = ""
    for msg in messages:
        formatted += f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>\n"
    formatted += "<|im_start|>assistant\n"  # 提示模型开始生成
    return formatted

print("ChatML 格式输出:")
print(format_chatml(sft_example["messages"]))

# loss mask 生成
def create_loss_mask(token_ids, assistant_start_positions):
    """
    仅在 assistant 部分的 token 上计算损失
    assistant_start_positions: 每个 assistant 回复的起始位置列表
    """
    mask = torch.zeros(len(token_ids))
    for start in assistant_start_positions:
        mask[start:] = 1  # 简化：从第一个 assistant token 开始
    return mask

print(f"\nToken IDs: {list(range(20))}")
print(f"Loss Mask: {create_loss_mask(torch.arange(20), [10]).int().tolist()}")


## 8.4 RLHF 与 DPO：让模型符合人类偏好

SFT 让模型"会回答"，但回答的质量、安全性、有用性仍可能不佳。
RLHF 和 DPO 旨在让模型的输出**符合人类偏好** (Human Alignment)。

### RLHF 三阶段

```
阶段1: SFT → 阶段2: 奖励模型训练 → 阶段3: PPO 强化学习
```

**阶段 2：奖励模型 (Reward Model, RM)**

- 对同一个 prompt，SFT 模型生成多个回答
- 人类标注员对这些回答**排序**（A > B > C > D）
- 训练一个模型，输入 (prompt, response)，输出**标量奖励** $r$
- 损失函数 (Bradley-Terry 模型)：
  $$\mathcal{L}_{\text{RM}} = -\log \sigma(r_\text{chosen} - r_\text{rejected})$$

**阶段 3：PPO (Proximal Policy Optimization)**

- 使用阶段 2 的 RM 作为奖励信号
- PPO 优化策略（LLM），同时加 KL 惩罚防止偏离 SFT 太远：
  $$\mathcal{L}_{\text{PPO}} = \mathbb{E}[r(x, y) - \beta \cdot \text{KL}(\pi_\theta \| \pi_{\text{ref}})]$$

### DPO：绕过奖励模型的直接偏好优化

RLHF 需要训练独立的奖励模型，工程复杂。DPO (Direct Preference Optimization, 2023) 提出直接
在偏好数据上优化策略，**无需训练奖励模型**：

$$
\mathcal{L}_{\text{DPO}} = -\log \sigma\left(\beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)
$$

其中 $y_w$ 是优选回答，$y_l$ 是劣选回答，$\pi_{\text{ref}}$ 是参考策略（通常为 SFT 模型）。

**直觉：** 提高优选回答相对于参考策略的概率，降低劣选回答的相对概率。


In [ ]:
# === DPO 损失函数实现 ===
def dpo_loss(pi_logps_chosen, pi_logps_rejected,
             ref_logps_chosen, ref_logps_rejected,
             beta=0.1):
    """
    计算 DPO 损失
    
    参数:
        pi_logps_chosen: 当前策略对优选回答的 log 概率
        pi_logps_rejected: 当前策略对劣选回答的 log 概率
        ref_logps_chosen: 参考策略对优选回答的 log 概率
        ref_logps_rejected: 参考策略对劣选回答的 log 概率
        beta: 温度参数（控制偏离参考策略的程度）
    """
    pi_log_ratio = pi_logps_chosen - pi_logps_rejected
    ref_log_ratio = ref_logps_chosen - ref_logps_rejected
    
    # DPO 核心公式
    loss = -F.logsigmoid(beta * (pi_log_ratio - ref_log_ratio))
    return loss.mean()

# 演示
# 模拟：当前策略稍微偏好 chosen，参考策略也偏好 chosen
pi_chosen = torch.randn(8) + 0.5   # chosen 稍高
pi_rejected = torch.randn(8) - 0.5
ref_chosen = torch.randn(8) + 0.3
ref_rejected = torch.randn(8) - 0.3

loss = dpo_loss(pi_chosen, pi_rejected, ref_chosen, ref_rejected)
print(f"DPO Loss: {loss.item():.4f}")

# 对比不同情况
print("\n=== 场景对比 ===")
# 1. 策略强烈偏好 chosen（理想）
l1 = dpo_loss(torch.tensor([-1.0]), torch.tensor([-5.0]),
              torch.tensor([-2.0]), torch.tensor([-4.0]))
print(f"策略偏好 chosen: loss={l1.item():.4f}")

# 2. 策略错误地偏好 rejected（糟糕）
l2 = dpo_loss(torch.tensor([-5.0]), torch.tensor([-1.0]),
              torch.tensor([-2.0]), torch.tensor([-4.0]))
print(f"策略偏好 rejected: loss={l2.item():.4f} (应更高)")

# 3. 策略与参考策略一致（无偏好变化）
l3 = dpo_loss(torch.tensor([-2.0]), torch.tensor([-4.0]),
              torch.tensor([-2.0]), torch.tensor([-4.0]))
print(f"与参考一致: loss={l3.item():.4f} (~0.693, sigmoid(0)=0.5 → -log(0.5))")


## 8.5 使用 HuggingFace 加载和推理 LLM

HuggingFace 的 `transformers` 库提供了统一的 API 访问数千个预训练模型。


In [ ]:
# === 使用 HuggingFace pipeline（最简单）===
# 注意：需要安装 transformers 库
# pip install transformers torch

# 轻量级演示：使用小型 GPT-2
try:
    from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
    
    # 方式 1：pipeline（一行代码）
    generator = pipeline('text-generation', model='distilgpt2', max_new_tokens=50)
    result = generator("The future of artificial intelligence is", 
                       num_return_sequences=1,
                       temperature=0.8)
    print("Pipeline 输出:")
    print(result[0]['generated_text'])
    
except ImportError:
    print("未安装 transformers 库。安装命令: pip install transformers")
except Exception as e:
    print(f"加载模型时出错: {e}")
    print("提示: 如果网络不通，可设置 HF_ENDPOINT=https://hf-mirror.com")


In [ ]:
# === 手动加载模型与 Tokenizer ===
try:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    
    model_name = "distilgpt2"
    
    # 加载 tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token  # GPT-2 没有 pad_token
    
    # 加载模型
    model = AutoModelForCausalLM.from_pretrained(model_name)
    
    # Tokenize
    prompt = "Explain backpropagation in simple terms:"
    inputs = tokenizer(prompt, return_tensors='pt')
    
    # 生成
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("生成结果:")
    print(generated_text)
    
except ImportError:
    print("未安装 transformers。安装: pip install transformers")
except Exception as e:
    print(f"错误: {e}")


### 常用生成参数

| 参数 | 含义 | 典型值 |
|------|------|--------|
| `temperature` | 控制随机性：越低越确定，越高越有创意 | 0.7 |
| `top_p` | Nucleus sampling：从累积概率 ≥ p 的 token 中采样 | 0.9 |
| `top_k` | 仅从概率最高的 k 个 token 中采样 | 50 |
| `max_new_tokens` | 最大生成长度 | 512 |
| `repetition_penalty` | 惩罚重复 token | 1.1 |
| `do_sample` | True=采样，False=贪心 | True |


## 本章小结

| 概念 | 要点 |
|------|------|
| **Transformer** | Self-Attention + FFN + LayerNorm，并行处理序列 |
| **Attention** | $\text{softmax}(QK^T/\sqrt{d_k})V$，捕捉 token 间关系 |
| **GPT 预训练** | Next token prediction，因果掩码 |
| **SFT** | 在指令-回答对上训练，loss mask 只计算回答部分 |
| **RLHF** | 奖励模型 + PPO，让模型符合人类偏好 |
| **DPO** | 绕过 RM，直接在偏好对比上优化策略 |
| **HuggingFace** | `pipeline` / `AutoModel` 统一加载和推理 |

✅ 第八章完成！你现在理解 LLM 从预训练到对齐的完整技术栈。
